# NB9 · Deployed model

**Machine Learning I · Group Project 2026-27 · The Guild of Gentle Landings**

Group XX · Data Avionaries

---

The final model chosen in NB1, and nothing else (no exploration, no comparison, no search). This notebook:
1. reads `learn.csv` and `predict.csv`;
2. applies the final preparation steps (the same `clean()` and `engineer()` as NB1);
3. trains the final model on all of `learn.csv`, with its hyperparameters written explicitly;
4. writes `MLXX_predictions.csv` in the Kaggle format.

Every source of randomness is fixed (`random_state=5`), so running it from top to bottom always produces the same file.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

print("python      ", sys.version.split()[0])
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("scikit-learn", sklearn.__version__)

GROUP = "XX"       # two-digit group number
SEED = 5
THRESHOLD = 0.50   # decision threshold chosen in NB1 (section 9)

python       3.13.18
numpy        2.5.3
pandas       3.0.5
scikit-learn 1.9.1


## 1. Read the data

In [2]:
learn = pd.read_csv("learn.csv")
predict = pd.read_csv("predict.csv")

X = learn.drop(columns=["row_id", "fatal"])
y = learn["fatal"]
X_predict = predict.drop(columns=["row_id"])
print(f"learn: {X.shape} | predict: {X_predict.shape}")

learn: (16979, 40) | predict: (4245, 40)


## 2. Preparation (same rules as NB1, sections 3-4)

In [3]:
def clean(df):
    """Rule-based cleaning: every rule looks at one row at a time, nothing is learned from the data."""
    df = df.copy()

    # 1. inconsistent spellings (air / AIR, n / N, ...)
    for col in ["acft_category", "homebuilt", "second_pilot", "acft_make"]:
        df[col] = df[col].str.upper().str.strip()

    # 2. columns with no useful information (see NB0, findings 2-5)
    df = df.drop(columns=["ev_country", "wx_temp_c", "gust_ind", "wind_dir_deg"])

    # 3. temperature and dew point: 0/0 means missing, impossible values -> NaN
    both_zero = (df["wx_temp"] == 0) & (df["wx_dew_pt"] == 0)
    df.loc[both_zero, ["wx_temp", "wx_dew_pt"]] = np.nan
    df.loc[(df["wx_temp"] > 130) | (df["wx_temp"] < -60), "wx_temp"] = np.nan
    df.loc[(df["wx_dew_pt"] > 100) | (df["wx_dew_pt"] < -60), "wx_dew_pt"] = np.nan

    # 4. altimeter: ~3000 is written without the decimal point; keep only 27-32 inHg (0 = missing)
    alt = df["altimeter"].where(df["altimeter"] < 2800, df["altimeter"] / 100)
    df["altimeter"] = alt.where(alt.between(27, 32))

    # 5. visibility is reported up to 10 statute miles
    df["vis_sm"] = df["vis_sm"].clip(upper=10)

    # 6. impossible weather and location values -> NaN
    df.loc[df["wind_vel_kts"] > 100, "wind_vel_kts"] = np.nan
    df.loc[df["gust_kts"] > 100, "gust_kts"] = np.nan
    df.loc[df["sky_ceil_ht"] > 40000, "sky_ceil_ht"] = np.nan
    df.loc[df["apt_dist"] > 500, "apt_dist"] = np.nan
    df.loc[df["apt_elev"] > 15000, "apt_elev"] = np.nan

    # 7. aircraft: impossible years and seat counts -> NaN
    bad_year = (df["acft_year"] < 1900) | (df["acft_year"] > df["ev_year"])
    df.loc[bad_year, "acft_year"] = np.nan
    df.loc[(df["total_seats"] == 0) | (df["total_seats"] > 600), "total_seats"] = np.nan

    # 8. engine power: horsepower only for non-jet engines (jets are in pounds of thrust)
    jet = df["eng_type"].isin(["TF", "TJ", "GTFN"])
    df["eng_power_hp"] = df["eng_power"].where(~jet)
    df.loc[(df["eng_power_hp"] == 0) | (df["eng_power_hp"] > 5000), "eng_power_hp"] = np.nan
    df = df.drop(columns="eng_power")

    # 9. pilot age outside 14-100 is a typing error
    df.loc[(df["pilot_age"] < 14) | (df["pilot_age"] > 100), "pilot_age"] = np.nan

    # 10. far_part codes that are not in the data dictionary -> UNK
    known = ["091", "137", "135", "121", "PUBU", "133", "129", "091K", "NUSN", "NUSC", "125"]
    df.loc[df["far_part"].notna() & ~df["far_part"].isin(known), "far_part"] = "UNK"
    return df

In [4]:
def engineer(df):
    """New features built from the cleaned columns (one row at a time, nothing learned from the data)."""
    df = df.copy()
    df["acft_age"] = df["ev_year"] - df["acft_year"]                     # age of the aircraft at the accident
    df["temp_dew_spread"] = df["wx_temp"] - df["wx_dew_pt"]              # small spread = humid air, fog risk
    df["night"] = df["light_cond"].isin(["NITE", "NDRK", "NBRT"]).astype(int)
    df["imc"] = (df["wx_cond_basic"] == "IMC").astype(int)               # instrument meteorological conditions
    df["off_airport"] = (df["apt_dist"] > 0).astype(float).where(df["apt_dist"].notna())
    df["log_apt_dist"] = np.log1p(df["apt_dist"])                        # distance is very skewed
    df["has_ceiling"] = (df["sky_ceil_ht"] > 0).astype(int)
    df["has_gust"] = (df["gust_kts"] > 0).astype(int)
    df["make"] = df["acft_make"].str.split().str[0]                      # ROBINSON HELICOPTER CO -> ROBINSON
    df = df.drop(columns=["acft_make", "acft_year"])
    return df

In [5]:
X_fe = engineer(clean(X))
X_predict_fe = engineer(clean(X_predict))

# features selected in NB1 (section 6)
num_features = ["ev_hour", "vis_sm", "wx_temp", "wx_dew_pt", "altimeter", "apt_elev", "num_eng", "total_seats",
                "pilot_age", "eng_power_hp", "acft_age", "temp_dew_spread", "night", "imc", "log_apt_dist", "has_ceiling"]
cat_features = ["light_cond", "wx_cond_basic", "sky_cond_ceil", "sky_cond_nonceil", "far_part", "acft_category",
                "homebuilt", "fixed_retractable", "type_fly", "flt_plan_filed", "second_pilot", "eng_type",
                "carb_fuel_injection", "pilot_med_certf", "phase", "make"]
print(len(num_features), "numerical +", len(cat_features), "categorical features")

16 numerical + 16 categorical features


## 3. Final model

In [6]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                      ("scaler", StandardScaler())]), num_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
                      ("onehot", OneHotEncoder(min_frequency=30, handle_unknown="infrequent_if_exist", sparse_output=False))]), cat_features),
])

model = RandomForestClassifier(n_estimators=300,
                               max_depth=None,
                               min_samples_leaf=5,
                               max_features="sqrt",
                               class_weight="balanced_subsample",
                               n_jobs=-1,
                               random_state=SEED)

final_pipeline = Pipeline([("pre", preprocessor), ("model", model)])
final_pipeline.fit(X_fe[num_features + cat_features], y)
print("trained on", len(X_fe), "accidents")

trained on 16979 accidents


## 4. Predict and write the submission

In [7]:
proba = final_pipeline.predict_proba(X_predict_fe[num_features + cat_features])[:, 1]
predictions = (proba >= THRESHOLD).astype(int)

submission = pd.DataFrame({"row_id": predict["row_id"], "fatal": predictions})

assert len(submission) == len(predict)
assert submission["fatal"].isin([0, 1]).all()

file_name = f"ML{GROUP}_predictions.csv"
submission.to_csv(file_name, index=False)
print(f"wrote {file_name}: {len(submission):,} rows | predicted fatal rate {submission['fatal'].mean():.1%}")
submission.head()

wrote MLXX_predictions.csv: 4,245 rows | predicted fatal rate 19.7%


,row_id,fatal
0,A00003,1
1,A00009,1
2,A00023,0
3,A00025,0
4,A00034,1


The predicted fatal rate (about 20%) is in line with the 17% observed in `learn.csv`: at this threshold the model flags slightly more accidents than are really fatal, as expected from its balance between precision and recall in NB1. This file is the one submitted to Kaggle and selected as final (saved as `MLXX_kaggle_submission.csv`).